# Stakeholder Gym — BOLD GRPO training on Kaggle T4

Bigger model (**Qwen 2.5-3B**), longer training (**200 steps**), higher LR (**2e-5**), larger dataset (L0+L1+L2 prompts), more GRPO samples per prompt (**6**). Targets a visibly-rising reward curve where the 0.5B/60-step run produced a flat one.

Kaggle notebook — set accelerator to **GPU T4 x2** (we use 1 GPU for training; the second can run a parallel smaller run if you want). Total wall time ~75 min.

## 0. GPU check — fail fast if missing

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), 'GPU not attached. Settings → Accelerator → GPU T4 x2.'
print(f'torch {torch.__version__}  cuda {torch.version.cuda}  GPUs {torch.cuda.device_count()}')
for i in range(torch.cuda.device_count()):
    print(f'  [{i}] {torch.cuda.get_device_name(i)}  {torch.cuda.get_device_properties(i).total_memory/1e9:.1f}GB')

## 1. Deps + clone (fast path)

Kaggle has torch preinstalled. Only install what's missing.

In [ ]:
!pip install -q unsloth trl datasets pydantic networkx pyyaml matplotlib

import os, subprocess
REPO = 'https://github.com/SAISriram19/meta.git'
WORK = '/kaggle/working/meta'
if not os.path.isdir(WORK):
    subprocess.run(['git', 'clone', '--depth', '1', REPO, WORK], check=True)
%cd {WORK}
!git log --oneline -1

## 2. Baseline eval — BEFORE numbers

Run only on L0+L2 to save time. L1+L3 numbers already in repo from hardened rerun.

In [ ]:
!python scripts/run_eval.py \
  --policies sycophant,keyword_principled,memory_aware \
  --scenarios L0_launch,L2_strategic_shift \
  --seeds 0,1,2 \
  --out eval_outputs/pre_train_kaggle

import json
with open('eval_outputs/pre_train_kaggle/summary.json') as f:
    pre = json.load(f)
print(json.dumps(pre, indent=2))

## 3. Load Qwen 2.5-3B with Unsloth 4-bit LoRA

3B in 4-bit = ~4GB VRAM base + activations. Fits in a single T4 (15GB) with room for long contexts.

In [ ]:
from unsloth import FastLanguageModel

MODEL_NAME = 'Qwen/Qwen2.5-3B-Instruct'
MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    target_modules=['q_proj','k_proj','v_proj','o_proj','gate_proj','up_proj','down_proj'],
    lora_alpha=64,
    lora_dropout=0.0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=42,
)
tokenizer.pad_token_id = tokenizer.pad_token_id or tokenizer.eos_token_id
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f'trainable: {trainable/1e6:.2f}M / total: {total/1e6:.2f}M')

## 4. Big prompt dataset — L0+L1+L2 observations

More prompts = more diverse GRPO batches = less noise in the reward curve.

In [ ]:
import sys, json
sys.path.insert(0, '.')
from env.environment import StakeholderEnv
from env.models import WaitAction
from scripts.train import SYSTEM_PROMPT, format_prompt, parse_completion

prompts = []
for scenario in ['L0_launch', 'L1_product_recall', 'L2_strategic_shift']:
    for _ in range(10):
        env = StakeholderEnv()
        obs = env.reset(task_id=scenario)
        for _ in range(8):
            if obs.new_messages:
                prompts.append(SYSTEM_PROMPT + '\n\nOBSERVATION:\n' + format_prompt(obs, env))
            r = env.step(WaitAction())
            obs = r.observation
            if r.done:
                break

from datasets import Dataset
ds = Dataset.from_dict({'prompt': prompts})
print(f'dataset: {len(ds)} prompts, avg prompt len: {sum(len(p) for p in prompts)//len(prompts)} chars')

## 5. Env-backed reward — sanity check discrimination

In [ ]:
def reward_fn(prompts, completions, **_):
    rewards = []
    for p, c in zip(prompts, completions):
        env = StakeholderEnv()
        env.reset(task_id='L0_launch')
        action = parse_completion(c, env)
        try:
            r = env.step(action)
            rewards.append(float(r.reward))
        except Exception:
            rewards.append(-0.1)
    return rewards

good = json.dumps({'type':'send_message','stakeholder_id':'alice','content':'I hear urgency but we need full QA','stance':'pushback','reasoning':'fabricated urgency authority appeal'})
bad  = json.dumps({'type':'send_message','stakeholder_id':'alice','content':'Sure, agreed','stance':'agree'})
wait = json.dumps({'type':'wait'})
rs = reward_fn([prompts[0]]*3, [good, bad, wait])
print(f'principled={rs[0]:.3f}  sycophant={rs[1]:.3f}  wait={rs[2]:.3f}')
assert rs[0] > rs[1], 'reward signal INVERTED — abort'

## 6. GRPO — 200 steps, LR 2e-5, 6 generations per prompt

Bold settings vs Colab 0.5B run:
- `max_steps=200` (vs 60): more gradient updates
- `learning_rate=2e-5` (vs 5e-6): 4x — 3B base can tolerate it
- `num_generations=6` (vs 4): tighter reward signal per prompt
- `max_completion_length=200` (vs 120): room for valid JSON
- `temperature=0.9` (vs 0.8): more exploration

In [ ]:
from trl import GRPOConfig, GRPOTrainer

config = GRPOConfig(
    output_dir='/kaggle/working/outputs/grpo-stakeholder',
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-5,
    max_prompt_length=1500,
    max_completion_length=200,
    num_generations=6,
    max_steps=200,
    logging_steps=1,
    save_steps=100,
    seed=42,
    temperature=0.9,
    beta=0.04,
    report_to='none',
)

trainer = GRPOTrainer(
    model=model,
    tokenizer=tokenizer,
    reward_funcs=[reward_fn],
    train_dataset=ds,
    args=config,
)
trainer.train()
# LoRA-only save (per guide §16 — do NOT naively merge 4-bit → 16-bit).
model.save_pretrained('/kaggle/working/outputs/grpo-stakeholder-lora')
tokenizer.save_pretrained('/kaggle/working/outputs/grpo-stakeholder-lora')
print('training done')

## 7. Reward curve — the pitch artifact

In [ ]:
import matplotlib.pyplot as plt
import os
os.makedirs('/kaggle/working/outputs', exist_ok=True)

log = trainer.state.log_history
steps = [h['step'] for h in log if 'reward' in h]
rewards = [h['reward'] for h in log if 'reward' in h]

plt.figure(figsize=(10, 4))
plt.plot(steps, rewards, label='mean reward per batch', marker='o', alpha=0.4, markersize=3)
window = 10
if len(rewards) >= window:
    ma = [sum(rewards[max(0,i-window+1):i+1])/min(i+1,window) for i in range(len(rewards))]
    plt.plot(steps, ma, label=f'{window}-step moving avg', linewidth=2.5, color='C1')
plt.xlabel('training step')
plt.ylabel('mean reward')
plt.title(f'Qwen 2.5-3B GRPO — anti-sycophancy reward ({len(rewards)} steps)')
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()
plt.savefig('/kaggle/working/outputs/grpo_reward_curve_3b.png', dpi=120)
plt.show()
print(f'first: {rewards[0]:.4f}  last: {rewards[-1]:.4f}  delta: {rewards[-1]-rewards[0]:+.4f}')
print(f'saved: /kaggle/working/outputs/grpo_reward_curve_3b.png')

## 8. Post-train eval — AFTER numbers on L0 + L2

In [ ]:
from eval.harness import EvalConfig, run_eval
from eval.policies import LLM_SYSTEM_PROMPT

FastLanguageModel.for_inference(model)

def make_trained_policy():
    def act(ctx):
        obs_json = format_prompt(ctx.observation, ctx.env)
        prompt = LLM_SYSTEM_PROMPT + '\n\nOBSERVATION:\n' + obs_json + '\n\nReturn ONE action as strict JSON.'
        inputs = tokenizer(prompt, return_tensors='pt', truncation=True, max_length=1800).to(model.device)
        out = model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=True,
            temperature=0.4,
            pad_token_id=tokenizer.eos_token_id,
        )
        text = tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        return parse_completion(text, ctx.env)
    return act

config_eval = EvalConfig(
    policies={'trained_qwen3b': make_trained_policy()},
    scenarios=['L0_launch', 'L2_strategic_shift'],
    seeds=[0, 1, 2],
    out_dir='eval_outputs/post_train_kaggle',
)
run_eval(config_eval, verbose=True)

## 9. Before / After comparison

In [ ]:
import json
with open('eval_outputs/pre_train_kaggle/summary.json') as f:
    pre = json.load(f)
with open('eval_outputs/post_train_kaggle/summary.json') as f:
    post = json.load(f)

print(f'{"policy":<24} {"scenario":<22} {"reward":>8}  {"syc_rate":>8}  {"bad":>5}  {"principled":>10}')
print('-' * 90)
for row in pre:
    print(f'BEFORE {row["policy"]:<18} {row["scenario"]:<22} {row["reward_mean"]:>+8.3f}  {row["sycophancy_rate"]:>8.3f}  {row["bad_agreements_mean"]:>5.1f}  {row["principled_pushbacks_mean"]:>10.1f}')
print()
for row in post:
    print(f' AFTER {row["policy"]:<18} {row["scenario"]:<22} {row["reward_mean"]:>+8.3f}  {row["sycophancy_rate"]:>8.3f}  {row["bad_agreements_mean"]:>5.1f}  {row["principled_pushbacks_mean"]:>10.1f}')

## 10. Save artifacts

Everything in `/kaggle/working/outputs/` is downloadable via the right-side panel under **Output** once the notebook finishes. Key files:
- `grpo_reward_curve_3b.png` — training curve
- `grpo-stakeholder-lora/` — LoRA adapters (tar it for download)

Also write pre/post summaries to the download path.

In [ ]:
import shutil, tarfile, json
out = '/kaggle/working/outputs'
# copy eval summaries so they're alongside the curve
shutil.copy('eval_outputs/pre_train_kaggle/summary.json', f'{out}/pre_train_summary.json')
shutil.copy('eval_outputs/post_train_kaggle/summary.json', f'{out}/post_train_summary.json')
# tar the LoRA adapters
with tarfile.open(f'{out}/grpo-stakeholder-lora.tar.gz', 'w:gz') as t:
    t.add(f'{out}/grpo-stakeholder-lora', arcname='grpo-stakeholder-lora')
print('outputs:')
for f in sorted(os.listdir(out)):
    print(f'  {f}')